In [5]:
#!/usr/bin/env python
# coding: utf-8

import os
import pandas as pd
import numpy as np
from glob import glob

# ====================================
# CONFIGURATION
# ====================================
BASE_DIR = "CRSS"  # path to main CRSS folder
OUTPUT_PATH = "cleaned_data/crss_accident_cleaned.csv"

VARS_OF_INTEREST = [
    "CASENUM", "VE_TOTAL", "REGION", "URBANICITY",
    "MONTH", "YEAR", "DAY_WEEK", "HOUR",
    "RELJCT2", "TYP_INT", "LGT_COND", "WEATHER"
]

# ====================================
# LOAD CRSS ACCIDENT FILE
# ====================================
def load_crss_accident_file(year_folder):
    """
    Load the CRSS accident file for a given year folder,
    selecting only the columns in VARS_OF_INTEREST to avoid duplicates.
    """
    # Handle capitalization differences
    candidates = glob(os.path.join(year_folder, "[Aa][Cc][Cc][Ii][Dd][Ee][Nn][Tt].csv"))
    if not candidates:
        print(f"No accident file found in {year_folder}")
        return None
    file_path = candidates[0]

    # Determine which columns exist and read only those
    try:
        df = pd.read_csv(file_path, usecols=VARS_OF_INTEREST, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, usecols=VARS_OF_INTEREST, encoding="latin1", low_memory=False)
    except ValueError:
        # Some columns might be missing; read intersection
        df_cols = pd.read_csv(file_path, nrows=0).columns.str.upper()
        cols_to_read = [c for c in VARS_OF_INTEREST if c in df_cols]
        df = pd.read_csv(file_path, usecols=cols_to_read, encoding="utf-8", low_memory=False)

    df.columns = df.columns.str.upper().str.strip()
    df["YEAR"] = int(os.path.basename(year_folder))
    return df

# ====================================
# CLEANING FUNCTIONS
# ====================================
def clean_crss_accident_df(df):
    """Apply cleaning steps to CRSS accident data."""
    df = df.copy()

    if "HOUR" in df.columns:
        df["HOUR"] = df["HOUR"].replace(99, pd.NA)

    for col, missing_codes in [("RELJCT2",[98,99]),
                               ("TYP_INT",[98,99]),
                               ("LGT_COND",[8,9]),
                               ("WEATHER",[98,99])]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col].isin(missing_codes), col] = pd.NA

    # Unique ID
    df["ID"] = "CRSS_" + df["YEAR"].astype(str) + "_" + df["CASENUM"].astype(str)
    df.insert(0, "ID", df.pop("ID"))

    return df

# ====================================
# COMBINE ALL YEARS
# ====================================
def combine_crss_accident_data(base_dir):
    year_folders = sorted([f.path for f in os.scandir(base_dir) if f.is_dir()])
    all_dfs = []

    for folder in year_folders:
        df = load_crss_accident_file(folder)
        if df is None:
            continue

        # Clean
        df = clean_crss_accident_df(df)
        all_dfs.append(df)
        print(f"Processed {os.path.basename(folder)}: {df.shape[0]} rows, {df.shape[1]} cols")

    combined = pd.concat(all_dfs, ignore_index=True)
    return combined

# ====================================
# DATA QUALITY CHECK
# ====================================
def summarize_data_quality(df):
    print("\n=== DATA QUALITY SUMMARY ===")
    missing_pct = df.isna().mean() * 100
    print("Missingness (%):")
    print(missing_pct.sort_values(ascending=False))
    print("\nRanges / Unique Values:")
    for col in df.columns:
        if df[col].dtype in [np.int64, np.float64]:
            print(f"{col}: min={df[col].min()}, max={df[col].max()}")
        else:
            print(f"{col}: {df[col].nunique()} unique values")

# ====================================
# MAIN SCRIPT
# ====================================
if __name__ == "__main__":
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    crss_accident_clean = combine_crss_accident_data(BASE_DIR)
    summarize_data_quality(crss_accident_clean)
    crss_accident_clean.to_csv(OUTPUT_PATH, index=False)
    print(f"\nCleaned CRSS accident data saved to: {OUTPUT_PATH}")


Processed 2016: 46511 rows, 13 cols
Processed 2017: 54969 rows, 13 cols
Processed 2018: 48443 rows, 13 cols
Processed 2019: 54409 rows, 13 cols
Processed 2020: 54745 rows, 13 cols
Processed 2021: 54200 rows, 13 cols
Processed 2022: 53955 rows, 13 cols
Processed 2023: 50103 rows, 13 cols

=== DATA QUALITY SUMMARY ===
Missingness (%):
TYP_INT       7.520577
RELJCT2       4.793272
WEATHER       4.382570
LGT_COND      0.708064
HOUR          0.527873
ID            0.000000
CASENUM       0.000000
VE_TOTAL      0.000000
MONTH         0.000000
YEAR          0.000000
DAY_WEEK      0.000000
URBANICITY    0.000000
REGION        0.000000
dtype: float64

Ranges / Unique Values:
ID: 417335 unique values
CASENUM: min=201600014311, max=202305779265
VE_TOTAL: min=1, max=15
MONTH: min=1, max=12
YEAR: min=2016, max=2023
DAY_WEEK: min=1, max=7
HOUR: 24 unique values
RELJCT2: min=1.0, max=20.0
TYP_INT: min=1.0, max=11.0
LGT_COND: min=1.0, max=7.0
WEATHER: min=1.0, max=12.0
URBANICITY: min=1, max=2
REGION: 